# Fix Raw Response JSON Parsing Errors

This notebook fixes JSON formatting errors in `raw_response` fields from Claude extraction results.

**Common Problems:**

1. **Unescaped quotes** inside string values:
```
"description_of_work": "... Network ("IFPN"). The source..."
```
Should be:
```
"description_of_work": "... Network (\"IFPN\"). The source..."
```

2. **Literal newlines** within arrays:
```
"account_numbers": ["1210-1-3400-5310",
"10"]
```
Should be:
```
"account_numbers": ["1210-1-3400-5310", "10"]
```

## 1. Setup

In [40]:
import json
import re
from pathlib import Path
from typing import Dict, Any, List

# Configuration
jloc = '../../../'
OUTPUT_DIR = Path(jloc + 'data/intermediate_products/eds_claude_json_production')

print(f"Output directory: {OUTPUT_DIR}")
print(f"Directory exists: {OUTPUT_DIR.exists()}")

Output directory: ../../../data/intermediate_products/eds_claude_json_production
Directory exists: True


## 2. Quote Fixing Function

In [ ]:
def fix_json_newlines(json_str: str) -> str:
    """
    Fix literal newlines within JSON arrays and strings.
    
    Common issues:
    1. Newlines in arrays: ["value1",\n"value2"] -> ["value1", "value2"]
    2. Newlines in strings: "text\nmore" -> "text\\nmore"
    """
    # First, remove newlines that appear within arrays between elements
    # Pattern: comma followed by newline followed by quote/bracket/brace
    json_str = re.sub(r',\s*\n\s*"', r', "', json_str)
    json_str = re.sub(r',\s*\n\s*\[', r', [', json_str)
    json_str = re.sub(r',\s*\n\s*\{', r', {', json_str)
    
    # Remove newlines after opening brackets/braces (within arrays/objects)
    json_str = re.sub(r'\[\s*\n\s*"', r'["', json_str)
    json_str = re.sub(r'\{\s*\n\s*"', r'{"', json_str)
    
    return json_str


def fix_json_quotes(json_str: str) -> str:
    """
    Fix unescaped quotes inside JSON string values.
    
    Handles the specific pattern where Claude returns text like:
    "... Network ("IFPN"). The source..."
    
    Strategy: Manually parse and escape quotes inside string values.
    """
    result = []
    i = 0
    in_string = False
    current_quote_is_value = False
    escape_next = False
    
    while i < len(json_str):
        char = json_str[i]
        
        if escape_next:
            result.append(char)
            escape_next = False
            i += 1
            continue
        
        if char == '\\':
            result.append(char)
            escape_next = True
            i += 1
            continue
        
        if char == '"':
            if not in_string:
                # Starting a string - determine if key or value
                in_string = True
                # Look back to see if this follows a colon (value) or comma/brace (key)
                j = i - 1
                while j >= 0 and json_str[j] in ' \n\t':
                    j -= 1
                if j >= 0 and json_str[j] == ':':
                    current_quote_is_value = True
                else:
                    current_quote_is_value = False
                result.append(char)
            else:
                # Potentially ending a string - check if this is really the end
                j = i + 1
                while j < len(json_str) and json_str[j] in ' \n\t':
                    j += 1
                
                # If we're in a value and next char is NOT a structural character,
                # this is an internal quote that should be escaped
                if current_quote_is_value and j < len(json_str) and json_str[j] not in ':,}]':
                    # Internal quote - escape it
                    result.append('\\"')
                else:
                    # Closing quote
                    result.append(char)
                    in_string = False
        else:
            result.append(char)
        
        i += 1
    
    return ''.join(result)


def fix_json_combined(json_str: str) -> str:
    """
    Apply all JSON fixes in sequence.
    """
    # Fix newlines first (structural issue)
    json_str = fix_json_newlines(json_str)
    # Then fix quotes (content issue)
    json_str = fix_json_quotes(json_str)
    return json_str


print("Quote and newline fixing functions loaded.")

## 3. Test the Fixing Function

In [ ]:
# Test with problematic examples

print("TEST 1: Unescaped quotes")
print("="*60)
test_json_1 = '''{
    "description_of_work": "Network ("IFPN"). The source",
    "vendor_name": "Test Corp"
}'''

print("Original JSON (will fail to parse):")
print(test_json_1)

try:
    json.loads(test_json_1)
    print("✓ Parses OK (unexpected!)")
except json.JSONDecodeError as e:
    print(f"✗ Parse error (expected): {e}")

print("\nFixed JSON:")
fixed_1 = fix_json_combined(test_json_1)
print(fixed_1)

try:
    parsed = json.loads(fixed_1)
    print("✓ Parses OK!")
except json.JSONDecodeError as e:
    print(f"✗ Still fails: {e}")

print("\n" + "="*60)
print("TEST 2: Newlines in arrays")
print("="*60)
test_json_2 = '''{
    "account_numbers": ["1210-1-3400-5310",
"10"],
    "vendor_name": "Test Corp"
}'''

print("Original JSON (will fail to parse):")
print(test_json_2)

try:
    json.loads(test_json_2)
    print("✓ Parses OK (unexpected!)")
except json.JSONDecodeError as e:
    print(f"✗ Parse error (expected): {e}")

print("\nFixed JSON:")
fixed_2 = fix_json_combined(test_json_2)
print(fixed_2)

try:
    parsed = json.loads(fixed_2)
    print("✓ Parses OK!")
    print(f"Parsed data: {parsed}")
except json.JSONDecodeError as e:
    print(f"✗ Still fails: {e}")

## 4. Find Files with raw_response

In [46]:
print("Scanning for files with raw_response field...")
print("="*60)

all_files = list(OUTPUT_DIR.glob('*.json'))
files_with_raw = []

for json_file in all_files:
    # Skip batch metadata files
    if json_file.name.startswith('batch_'):
        continue
    
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        if 'raw_response' in data.get('extracted_data', {}):
            files_with_raw.append(json_file)
    except:
        continue

print(f"Total JSON files: {len(all_files)}")
print(f"Files with raw_response: {len(files_with_raw)}")

if files_with_raw:
    print(f"\nFirst 10 files:")
    for f in files_with_raw[:10]:
        print(f"  - {f.name}")

Scanning for files with raw_response field...
Total JSON files: 26287
Files with raw_response: 22

First 10 files:
  - 53011-001.json
  - 11014-003.json
  - 63492-000.json
  - 64978-001.json
  - 86829-000.json
  - 66569-000.json
  - 93867-000.json
  - 65961-000.json
  - 30615-001.json
  - 32547-008.json


## 5. Process and Fix Files

In [ ]:
def process_file(json_file: Path) -> tuple[bool, str]:
    """
    Process a single JSON file with raw_response field.
    Returns: (success: bool, message: str)
    """
    try:
        # Read the file
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        # Check if it has raw_response
        if 'raw_response' not in data.get('extracted_data', {}):
            return False, "No raw_response field"
        
        raw_response = data['extracted_data']['raw_response']
        
        # Try to parse directly first (maybe it's already OK)
        try:
            parsed = json.loads(raw_response)
            # Already parses - just clean up the structure
            data['extracted_data'].pop('error', None)
            data['extracted_data'].pop('raw_response', None)
            data['extracted_data'].update(parsed)
            
            with open(json_file, 'w') as f:
                json.dump(data, f, indent=2)
            
            return True, "Already valid - cleaned up"
        
        except json.JSONDecodeError as e:
            # Need to fix
            pass
        
        # Try to fix with combined fixer (newlines + quotes)
        fixed_json = fix_json_combined(raw_response)
        
        # Try to parse the fixed version
        try:
            parsed = json.loads(fixed_json)
            
            # Success! Update the data structure
            data['extracted_data'].pop('error', None)
            data['extracted_data'].pop('raw_response', None)
            data['extracted_data'].update(parsed)
            
            # Save back
            with open(json_file, 'w') as f:
                json.dump(data, f, indent=2)
            
            return True, "Fixed and parsed successfully"
        
        except json.JSONDecodeError as e2:
            return False, f"Fix failed: {e2}"
    
    except Exception as e:
        return False, f"Unexpected error: {e}"


print("File processing function loaded.")

In [49]:
# Process all files
if not files_with_raw:
    print("No files to process!")
else:
    print("Processing files...")
    print("="*60)
    
    success_count = 0
    fail_count = 0
    failed_files = []
    
    for i, json_file in enumerate(files_with_raw, 1):
        success, message = process_file(json_file)
        
        if success:
            success_count += 1
            if i <= 10 or i % 10 == 0:  # Show first 10 and every 10th
                print(f"{i}. ✓ {json_file.name}: {message}")
        else:
            fail_count += 1
            failed_files.append((json_file.name, message))
            print(f"{i}. ✗ {json_file.name}: {message}")
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Total files processed: {len(files_with_raw)}")
    print(f"Successfully fixed: {success_count}")
    print(f"Failed to fix: {fail_count}")
    
    if failed_files:
        print(f"\nFailed files:")
        for filename, msg in failed_files[:20]:  # Show first 20 failures
            print(f"  - {filename}: {msg}")
        if len(failed_files) > 20:
            print(f"  ... and {len(failed_files) - 20} more")
    
    print("="*60)

Processing files...
1. ✗ 53011-001.json: Fix failed: Invalid \escape: line 25 column 42 (char 1026)
2. ✗ 11014-003.json: Fix failed: Expecting property name enclosed in double quotes: line 25 column 277 (char 1317)
3. ✗ 63492-000.json: Fix failed: Expecting ',' delimiter: line 17 column 5 (char 739)
4. ✗ 64978-001.json: Fix failed: Invalid \escape: line 16 column 24 (char 658)
5. ✗ 86829-000.json: Fix failed: Expecting ',' delimiter: line 15 column 52 (char 679)
6. ✗ 66569-000.json: Fix failed: Expecting ',' delimiter: line 17 column 5 (char 719)
7. ✗ 93867-000.json: Fix failed: Expecting ',' delimiter: line 16 column 5 (char 667)
8. ✗ 65961-000.json: Fix failed: Invalid \escape: line 30 column 93 (char 1557)
9. ✗ 30615-001.json: Fix failed: Expecting property name enclosed in double quotes: line 26 column 209 (char 1430)
10. ✗ 32547-008.json: Fix failed: Invalid \escape: line 30 column 73 (char 1343)
11. ✗ 87312-000.json: Fix failed: Expecting ',' delimiter: line 15 column 36 (char 70

## 6. Verify Fixes

In [51]:
# Check how many files still have raw_response after fixing
print("Verifying fixes...")
print("="*60)

remaining_raw = []
for json_file in OUTPUT_DIR.glob('*.json'):
    if json_file.name.startswith('batch_'):
        continue
    
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        if 'raw_response' in data.get('extracted_data', {}):
            remaining_raw.append(json_file.name)
    except:
        continue

print(f"Files still with raw_response: {len(remaining_raw)}")

if remaining_raw:
    print(f"\nThese files still need manual attention:")
    for f in remaining_raw[:20]:
        print(f"  - {f}")
    if len(remaining_raw) > 20:
        print(f"  ... and {len(remaining_raw) - 20} more")
else:
    print("\n✓ All raw_response fields have been successfully processed!")

Verifying fixes...
Files still with raw_response: 22

These files still need manual attention:
  - 53011-001.json
  - 11014-003.json
  - 63492-000.json
  - 64978-001.json
  - 86829-000.json
  - 66569-000.json
  - 93867-000.json
  - 65961-000.json
  - 30615-001.json
  - 32547-008.json
  - 87312-000.json
  - 88059-000.json
  - 46873-000.json
  - 56518-001.json
  - 38893-000.json
  - 72485-001.json
  - 114239-000.json
  - 41810-001.json
  - 64772-000.json
  - 55723-004.json
  ... and 2 more
